# 03 — Treinamento de Deep Learning no PTB-XL

**DCA3606 — Inteligência Artificial (UFRN).** MLP, CNN 1D e ResNet 1D para classificação Normal × Alterado e regressão da idade. O notebook consome **somente** os artefatos da etapa 2; não recalcula alvos, partições ou normalização. Não há autoencoder nem Random Forest aqui.

Por segurança, `MODO_TESTE=True` é o padrão: executa uma época em pequenas amostras na CPU. O treinamento completo só é habilitado explicitamente no Google Colab. O conjunto de teste é conferido como metadado de partição, mas **nunca é enviado ao treinador nem usado para selecionar modelos**.

## 1. Caminhos e modo de execução

No Colab, transfira o projeto para o Google Drive e ajuste `RAIZ_PROJETO_COLAB` abaixo. Mantenha `PASTA_SAIDA` no Drive para persistir checkpoints, históricos e gráficos após desconexão. A cópia opcional dos dados preparados para `/content` acelera a leitura durante o treinamento; a origem e as saídas permanecem no Drive. O PTB-XL original em `records100/` **não é necessário** para treinar a partir da matriz preparada.

Para executar o treinamento completo: no Colab, configure `MODO_TESTE=False` e `EXECUTAR_TREINO_COMPLETO=True`. Fora do Colab, o código bloqueia o treinamento completo.

In [1]:
from pathlib import Path
import json
import shutil
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display

MODO_TESTE = True
EXECUTAR_TREINO_COMPLETO = False
RAIZ_PROJETO_COLAB = Path('/content/drive/MyDrive/ptbxl-projeto')  # ajuste no Colab
PASTA_DADOS_PREPARADOS_COLAB = RAIZ_PROJETO_COLAB / 'resultados' / 'preprocessamento'
PASTA_SAIDA_COLAB = RAIZ_PROJETO_COLAB / 'resultados' / 'deep_learning'
COPIAR_DADOS_PARA_CONTENT = True
SEMENTE = 3606

try:
    import google.colab
    EM_COLAB = True
except ModuleNotFoundError:
    EM_COLAB = False
if not MODO_TESTE and (not EM_COLAB or not EXECUTAR_TREINO_COMPLETO):
    raise RuntimeError('Treino completo permitido apenas no Colab com EXECUTAR_TREINO_COMPLETO=True.')
if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ_PROJETO = RAIZ_PROJETO_COLAB
else:
    RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / 'src' / 'deep_learning.py').is_file():
    raise FileNotFoundError(f'Módulo do projeto não encontrado em {RAIZ_PROJETO / "src"}')
sys.path.insert(0, str(RAIZ_PROJETO))

from src.deep_learning import (
    TAREFAS, MODELOS, ConfigTreino, criar_modelo, parametros_treinaveis,
    verificar_artefatos, selecionar_particoes, treinar_experimento
)

PASTA_DADOS_ORIGEM = (PASTA_DADOS_PREPARADOS_COLAB if EM_COLAB else
                      RAIZ_PROJETO / 'resultados' / 'preprocessamento')
PASTA_SAIDA_BASE = PASTA_SAIDA_COLAB if EM_COLAB else RAIZ_PROJETO / 'resultados' / 'deep_learning'
PASTA_SAIDA = PASTA_SAIDA_BASE / (
    'teste_rapido_v1' if MODO_TESTE else 'treino_completo_v1')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_DADOS = PASTA_DADOS_ORIGEM
if EM_COLAB and COPIAR_DADOS_PARA_CONTENT:
    PASTA_DADOS = Path('/content/ptbxl_preparado')
    PASTA_DADOS.mkdir(parents=True, exist_ok=True)
    for nome in ('normalizacao.json', 'manifesto.csv', 'classificacao.csv',
                 'regressao.csv', 'sinais_normalizados.float32.bin'):
        origem = PASTA_DADOS_ORIGEM / nome
        destino = PASTA_DADOS / nome
        if not destino.exists() or destino.stat().st_size != origem.stat().st_size:
            shutil.copy2(origem, destino)

dispositivo = torch.device('cpu' if MODO_TESTE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if MODO_TESTE:
    torch.set_num_threads(2)
torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
print('Modo teste:', MODO_TESTE, '| Colab:', EM_COLAB, '| dispositivo:', dispositivo)
print('Projeto:', RAIZ_PROJETO)
print('Dados preparados:', PASTA_DADOS)
print('Saídas persistentes:', PASTA_SAIDA)
print('PyTorch:', torch.__version__)

Modo teste: True | Colab: False | dispositivo: cpu
Projeto: /home/eduardo/Documentos/project-ia-dca
Dados preparados: /home/eduardo/Documentos/project-ia-dca/resultados/preprocessamento
Saídas persistentes: /home/eduardo/Documentos/project-ia-dca/resultados/deep_learning/teste_rapido_v1
PyTorch: 2.15.0+cpu


## 2. Conferência dos artefatos e das divisões

São verificados formato e tamanho da matriz, normalização ajustada nos folds 1–8, `strat_fold` e `particao`, ausência de pacientes cruzando partições, índices e alvos válidos. As três redes usam os mesmos subconjuntos de treino e validação em cada tarefa. Em `MODO_TESTE`, amostras pequenas são retiradas apenas dessas duas partições, com semente fixa; o teste permanece fechado.

In [2]:
parametros, tabelas = verificar_artefatos(PASTA_DADOS)
print('Forma da matriz preparada:', parametros['shape_sinais'])
for tarefa in TAREFAS:
    tabela = tabelas[tarefa]
    print(tarefa, '— folds oficiais:', tabela.groupby('particao').size().to_dict())
selecionados = {}
for tarefa in TAREFAS:
    treino, validacao = selecionar_particoes(tabelas[tarefa], tarefa, MODO_TESTE, SEMENTE)
    selecionados[tarefa] = pd.concat([treino, validacao], ignore_index=True)
    print(tarefa, '— amostras desta execução:',
          selecionados[tarefa].groupby('particao').size().to_dict())
    assert not selecionados[tarefa].particao.eq('teste').any()
print('Paciente compartilhado entre partições: 0 (verificação concluída).')

Forma da matriz preparada: [21789, 1000, 12]
classificacao — folds oficiais: {'teste': 2107, 'treino': 16730, 'validacao': 2105}
regressao — folds oficiais: {'teste': 2164, 'treino': 17200, 'validacao': 2141}
classificacao — amostras desta execução: {'treino': 32, 'validacao': 16}
regressao — amostras desta execução: {'treino': 32, 'validacao': 16}
Paciente compartilhado entre partições: 0 (verificação concluída).


## 3. Arquiteturas e hiperparâmetros

- **MLP:** vetor de 12.000 valores, camadas densas 256 e 128 com ReLU e dropout.
- **CNN 1D:** 12 canais, três convoluções temporais (32, 64 e 128 filtros), batch normalization, pooling e média global.
- **ResNet 1D:** stem temporal, seis blocos residuais com conexões de atalho, redução temporal e média global.

Cada rede termina com **uma saída escalar**. Na classificação, é um logit treinado com `BCEWithLogitsLoss`; na regressão, é uma idade contínua em anos treinada com `MSELoss`. A validação registra F1 macro ou MAE, além de outras métricas. O melhor checkpoint é escolhido pela **menor perda de validação**. Parâmetros, configuração e tempo são persistidos por experimento.

In [3]:
hiperparametros = {
    'mlp': {'taxa_aprendizado': 1e-3, 'dropout': 0.30},
    'cnn1d': {'taxa_aprendizado': 1e-3, 'dropout': 0.20},
    'resnet1d': {'taxa_aprendizado': 5e-4, 'dropout': 0.20},
}
EPOCAS = 1 if MODO_TESTE else 40
TAMANHO_LOTE = 8 if MODO_TESTE else 64
PACIENCIA = 2 if MODO_TESTE else 7
TRABALHADORES = 0 if MODO_TESTE else 2
configuracoes = {}
for tarefa in TAREFAS:
    for arquitetura in MODELOS:
        h = hiperparametros[arquitetura]
        configuracoes[(tarefa, arquitetura)] = ConfigTreino(
            tarefa=tarefa, arquitetura=arquitetura, epocas=EPOCAS,
            tamanho_lote=TAMANHO_LOTE, taxa_aprendizado=h['taxa_aprendizado'],
            decaimento_pesos=1e-4, dropout=h['dropout'], paciencia=PACIENCIA,
            delta_minimo=1e-4, semente=SEMENTE, trabalhadores=TRABALHADORES,
            modo_teste=MODO_TESTE)

display(pd.DataFrame([{
    'arquitetura': nome,
    'entrada': '(12.000,)' if nome == 'mlp' else '(12, 1.000)',
    'parâmetros treináveis': parametros_treinaveis(criar_modelo(nome, hiperparametros[nome]['dropout'])),
    'taxa de aprendizado': hiperparametros[nome]['taxa_aprendizado'],
    'dropout': hiperparametros[nome]['dropout']
} for nome in MODELOS]))
print('Épocas:', EPOCAS, '| lote:', TAMANHO_LOTE, '| paciência:', PACIENCIA)

,arquitetura,entrada,parâmetros treináveis,taxa de aprendizado,dropout
0,mlp,"(12.000,)",3105281,0.0010,0.3
1,cnn1d,"(12, 1.000)",38305,0.0010,0.2
2,resnet1d,"(12, 1.000)",242625,0.0005,0.2


Épocas: 1 | lote: 8 | paciência: 2


## 4. Treinamento com retomada

São executados seis experimentos independentes. Ao fim de **cada época**, `ultimo.pt` guarda pesos, estado do otimizador, gerador de embaralhamento, histórico e contador do early stopping; `melhor.pt` guarda o modelo de menor perda de validação. `historico.csv`, `curvas.png` e `configuracao.json` ficam na mesma pasta. Após uma desconexão, execute novamente o notebook com os mesmos caminhos e hiperparâmetros: cada experimento retoma da época seguinte ou reconhece que já terminou. Uma configuração diferente exige outra pasta de saída.

In [4]:
resultados_execucao = []
for tarefa in TAREFAS:
    for arquitetura in MODELOS:
        resultado = treinar_experimento(
            PASTA_DADOS, PASTA_SAIDA, selecionados[tarefa], parametros,
            configuracoes[(tarefa, arquitetura)], dispositivo)
        resultados_execucao.append(resultado)
resumo_execucao = pd.DataFrame(resultados_execucao)
display(resumo_execucao)
resumo_execucao.to_csv(PASTA_SAIDA / 'resumo_experimentos.csv', index=False)
print('Teste reservado: nenhuma época usou dados da partição teste.')

classificacao/mlp | época 1/1 | treino 0.6979 | validação 0.6776 | 0.2 s


classificacao/cnn1d | época 1/1 | treino 0.6803 | validação 0.6832 | 0.1 s


classificacao/resnet1d | época 1/1 | treino 0.6940 | validação 0.6960 | 0.2 s


regressao/mlp | época 1/1 | treino 3954.6130 | validação 3964.3561 | 0.2 s


regressao/cnn1d | época 1/1 | treino 3977.5644 | validação 4064.0723 | 0.1 s


regressao/resnet1d | época 1/1 | treino 3992.4003 | validação 4078.8385 | 0.3 s


,tarefa,arquitetura,epocas_executadas,melhor_perda_validacao,parametros_treinaveis,tempo_segundos,retomado_sem_treinar
0,classificacao,mlp,1,0.677576,3105281,0.184151,False
1,classificacao,cnn1d,1,0.683232,38305,0.101859,False
2,classificacao,resnet1d,1,0.695981,242625,0.243733,False
3,regressao,mlp,1,3964.356079,3105281,0.160372,False
4,regressao,cnn1d,1,4064.072266,38305,0.134614,False
5,regressao,resnet1d,1,4078.838501,242625,0.269898,False


Teste reservado: nenhuma época usou dados da partição teste.


## 5. Leitura dos resultados

Inspecione `resumo_experimentos.csv` e as curvas de cada pasta. As métricas exibidas durante o treinamento são de **validação**. A avaliação única no teste deve ocorrer em etapa posterior, depois da seleção final dos modelos, sempre carregando `melhor.pt`. Resultados de `MODO_TESTE` verificam apenas o funcionamento do código; não medem desempenho científico.

In [5]:
for tarefa in TAREFAS:
    for arquitetura in MODELOS:
        pasta = PASTA_SAIDA / tarefa / arquitetura
        historico = pd.read_csv(pasta / 'historico.csv')
        print(f'{tarefa}/{arquitetura}: {len(historico)} época(s), '
              f'melhor perda de validação = {historico.perda_validacao.min():.4f}, '
              f'tempo = {historico.segundos_epoca.sum():.1f} s')
        assert (pasta / 'melhor.pt').is_file()
        assert (pasta / 'ultimo.pt').is_file()
        assert (pasta / 'curvas.png').is_file()
print('Todos os seis experimentos geraram checkpoints, históricos e curvas.')

classificacao/mlp: 1 época(s), melhor perda de validação = 0.6776, tempo = 0.2 s
classificacao/cnn1d: 1 época(s), melhor perda de validação = 0.6832, tempo = 0.1 s
classificacao/resnet1d: 1 época(s), melhor perda de validação = 0.6960, tempo = 0.2 s
regressao/mlp: 1 época(s), melhor perda de validação = 3964.3561, tempo = 0.2 s
regressao/cnn1d: 1 época(s), melhor perda de validação = 4064.0723, tempo = 0.1 s
regressao/resnet1d: 1 época(s), melhor perda de validação = 4078.8385, tempo = 0.3 s
Todos os seis experimentos geraram checkpoints, históricos e curvas.


## Verificação local realizada

O `MODO_TESTE` foi executado na CPU para as **seis combinações** de tarefa e arquitetura: **uma época**, **32 ECGs de treino** e **16 de validação** por tarefa. Todas as redes produziram perdas finitas e salvaram `ultimo.pt`, `melhor.pt`, `historico.csv`, `curvas.png` e `configuracao.json`. A reabertura de um checkpoint concluído foi verificada sem repetir a época. Esses números apenas confirmam a execução do pipeline; **não são resultados de desempenho do projeto**. Nenhum sinal do fold 10 foi usado no treinamento.

## Como executar o treinamento completo no Google Colab

1. No Google Drive, crie uma pasta para o projeto (por exemplo, `MyDrive/ptbxl-projeto`). Transfira **`03_treinamento_deep_learning.ipynb`**, **`src/deep_learning.py`**, **`src/__init__.py`** e, dentro de **`resultados/preprocessamento/`**, os cinco arquivos: `normalizacao.json`, `manifesto.csv`, `classificacao.csv`, `regressao.csv` e `sinais_normalizados.float32.bin` (aproximadamente 0,974 GiB). O dataset bruto `records100/` não é necessário nesta etapa.
2. Abra o notebook no Colab. Em **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione GPU se disponível. Confirme que PyTorch, NumPy, pandas e Matplotlib estão disponíveis; se faltar algo, instale com `%pip install torch numpy pandas matplotlib`.
3. Na primeira célula de código, ajuste `RAIZ_PROJETO_COLAB`, `PASTA_DADOS_PREPARADOS_COLAB` e `PASTA_SAIDA_COLAB` para os diretórios do Drive. Defina `MODO_TESTE=False` e `EXECUTAR_TREINO_COMPLETO=True`. Execute as células em ordem. O dispositivo será CUDA quando disponível, caso contrário CPU.
4. Mantenha a pasta do projeto no Drive. Checkpoints, históricos, configurações e curvas serão gravados em `resultados/deep_learning/treino_completo_v1/`. Após uma desconexão, execute novamente com **a mesma configuração** para retomar. Para uma nova rodada com hiperparâmetros diferentes, use outra pasta de saída.

O treinamento completo **não foi executado nesta etapa**. O fold 10 permanece reservado para avaliação posterior.